# Phase 7 — OmniShift v2 (EWGS + PoT Activations)

**Base**: Phase 5 (EWGS) + Phase 6 (PoT Activations) combined  
**Change**: All quantizers use EWGS backward; PoTActivationEWGS after every ReLU

- Weights: W ∈ {0, ±2^p} via SparseShiftConv2dEWGS
- BN scales: γ/σ → ±2^q via PoTBatchNorm2dEWGS  
- Activations: post-ReLU → {0} ∪ {2^p} via PoTActivationEWGS

Fully multiply-free forward pass end-to-end.  
EWGS applied to all three quantization axes for smooth training.

Models:
- `omnishift_v2_fixed50` — fixed 50% sparsity
- `omnishift_v2_learnable` — learnable sparsity

**Hardware validation** (no FPGA required):
- FPGA resource estimation via theoretical Xilinx 7-series model
- TensorRT FP16 benchmark on T4 GPU

## Setup — clone repo and install dependencies

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/OmniShift'):
    subprocess.run(['git','clone','https://github.com/CryAndRRich/OmniShift.git','/kaggle/working/OmniShift'], check=True)
os.chdir('/kaggle/working/OmniShift')
!pip install -q pyyaml

## Verify GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU only'
print(f'Device : {gpu_name}')
print(f'PyTorch: {torch.__version__}')

## Training runs

### CIFAR-10 · fixed50 + EWGS + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase7_omnishift_v2.yaml --name omnishift_v2_fixed50 --dataset cifar10

### CIFAR-10 · learnable + EWGS + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase7_omnishift_v2.yaml --name omnishift_v2_learnable --dataset cifar10

### SVHN · fixed50 + EWGS + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase7_omnishift_v2.yaml --name omnishift_v2_fixed50 --dataset svhn

### SVHN · learnable + EWGS + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase7_omnishift_v2.yaml --name omnishift_v2_learnable --dataset svhn

## FPGA Resource Estimation

Theoretical Xilinx 7-series resource utilization. No physical FPGA required.

**Model**: Artix-7 XC7A200T (740 DSP48E2, 134k LUT)  
**Key claim**: Interior conv stack uses 0 DSP48E2 — all weights are bit-shifts (wired barrel-shifters).

In [ ]:
# Cross-phase FPGA comparison
!python scripts/fpga_estimate.py --baseline

In [ ]:
# Phase 7 learnable FPGA estimate
!python scripts/fpga_estimate.py --config configs/phase7_omnishift_v2.yaml --name omnishift_v2_learnable --dataset cifar10

## TensorRT Deployment Benchmark

Measures actual inference latency and throughput on T4 GPU via TensorRT FP16.  
This is a deployment speed validation — separate from the theoretical 45nm energy model.

In [ ]:
# Batch=1 (edge single-sample inference)
!python scripts/trt_benchmark.py \
    --config configs/phase7_omnishift_v2.yaml \
    --name omnishift_v2_learnable \
    --dataset cifar10 \
    --batch 1 --n-runs 1000

In [ ]:
# Batch=64 (throughput mode)
!python scripts/trt_benchmark.py \
    --config configs/phase7_omnishift_v2.yaml \
    --name omnishift_v2_learnable \
    --dataset cifar10 \
    --batch 64 --n-runs 500

## Results summary

In [ ]:
!python scripts/summarize_results.py